# 02 - Generate Teacher Pseudo Labels for YOLOPv2 Finetuning

This Colab notebook builds a mixed CARLA + real dashcam dataset without using YOLOPv2 as its own labeler.

Teachers used here:
- Bounding boxes: YOLOv8x (Ultralytics / COCO), mapped into YOLOPv2's driving classes.
- Lane masks: Ultra-Fast-Lane-Detection-v2 (CULane ResNet-18 checkpoint).
- Drivable area masks: SegFormer-B2 fine-tuned on Cityscapes.

Output images and masks are written at 320x320 so they match the YOLOPv2 finetuning/export target.

## 1 - Colab Setup

In [ ]:
%pip install -q ultralytics transformers accelerate gdown opencv-python pillow pandas matplotlib tqdm addict imagesize

import os
import sys
import math
import shutil
import subprocess
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

# Change this if your repo folder has a different Drive location.
PROJECT_ROOT = Path('/content/drive/MyDrive/sdcar-perception')
DATA_ROOT = PROJECT_ROOT / 'data'
CARLA_IMAGE_DIR = DATA_ROOT / 'carla' / 'images'
REAL_VIDEO_DIR = DATA_ROOT / 'real' / 'videos'
OUT_ROOT = DATA_ROOT / 'yolopv2_finetune'

IMAGE_SIZE = 320
TARGET_REAL_FRAMES = 5000
YOLO_CONF = 0.65
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

OUT_IMAGES = OUT_ROOT / 'images'
OUT_LABELS = OUT_ROOT / 'labels'
OUT_LANE = OUT_ROOT / 'masks' / 'lane'
OUT_DRIVABLE = OUT_ROOT / 'masks' / 'drivable'
OUT_QA = OUT_ROOT / 'qa_samples'
for d in (OUT_IMAGES, OUT_LABELS, OUT_LANE, OUT_DRIVABLE, OUT_QA):
    d.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('CARLA images:', CARLA_IMAGE_DIR)
print('Real videos:', REAL_VIDEO_DIR)
print('Output:', OUT_ROOT)
print('Device:', DEVICE)

## 2 - Inspect Dataset Structure

In [ ]:
def inspect_tree(root: Path, max_depth=3):
    rows = []
    root = Path(root)
    for path in sorted(root.rglob('*')):
        rel = path.relative_to(root)
        if len(rel.parts) > max_depth:
            continue
        if path.is_dir():
            kind, count = 'dir', sum(1 for _ in path.iterdir())
        else:
            kind, count = path.suffix.lower() or 'file', path.stat().st_size
        rows.append({'path': str(rel), 'kind': kind, 'count_or_bytes': count})
    return pd.DataFrame(rows)

summary = inspect_tree(DATA_ROOT)
display(summary)

carla_images = sorted(CARLA_IMAGE_DIR.glob('*.jpg')) + sorted(CARLA_IMAGE_DIR.glob('*.png'))
real_videos = sorted(REAL_VIDEO_DIR.glob('*.mp4')) + sorted(REAL_VIDEO_DIR.glob('*.mov')) + sorted(REAL_VIDEO_DIR.glob('*.avi'))
print(f'CARLA images: {len(carla_images)}')
print(f'Real videos: {len(real_videos)}')
for vp in real_videos:
    cap = cv2.VideoCapture(str(vp))
    fps = cap.get(cv2.CAP_PROP_FPS) or 0
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    cap.release()
    duration = frames / fps if fps else 0
    print(f'{vp.name}: {frames} frames, {fps:.2f} fps, {duration:.1f}s, {w}x{h}')

## 3 - Download and Load Teacher Models

In [ ]:
TEACHER_DIR = Path('/content/teacher_models')
TEACHER_DIR.mkdir(parents=True, exist_ok=True)

# YOLOv8x auto-downloads yolov8x.pt the first time this runs.
from ultralytics import YOLO
box_teacher = YOLO('yolov8x.pt')

# SegFormer-B2 Cityscapes road segmentation.
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
segformer_name = 'nvidia/segformer-b2-finetuned-cityscapes-1024-1024'
seg_processor = SegformerImageProcessor.from_pretrained(segformer_name)
seg_model = SegformerForSemanticSegmentation.from_pretrained(segformer_name).to(DEVICE).eval()
seg_id2label = {int(k): v.lower() for k, v in seg_model.config.id2label.items()}
ROAD_CLASS_IDS = [idx for idx, name in seg_id2label.items() if name == 'road']
if not ROAD_CLASS_IDS:
    raise RuntimeError(f'Could not find Cityscapes road class in {seg_id2label}')
print('SegFormer road ids:', ROAD_CLASS_IDS)

# UFLD-v2 official repo + CULane ResNet-18 teacher checkpoint.
UFLD_ROOT = Path('/content/Ultra-Fast-Lane-Detection-v2')
if not UFLD_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/cfzd/Ultra-Fast-Lane-Detection-v2', str(UFLD_ROOT)], check=True)

ufld_ckpt = TEACHER_DIR / 'ufldv2_culane_res18.pth'
if not ufld_ckpt.exists():
    # Official README Google Drive id for CULane ResNet-18.
    subprocess.run(['gdown', '--id', '1oEjJraFr-3lxhX_OXduAGFWalWa6Xh3W', '-O', str(ufld_ckpt)], check=True)

# Patch the cloned UFLD CULane model so inference does not import training-only
# DALI helpers through utils.common, and so CPU fallback works if Colab has no GPU.
model_culane_py = UFLD_ROOT / 'model' / 'model_culane.py'
model_culane_text = model_culane_py.read_text(encoding='utf-8')
if 'from utils.common import initialize_weights' in model_culane_text:
    init_helper = """

def initialize_weights(*models):
    for model in models:
        for m in model.modules():
            if isinstance(m, torch.nn.Conv2d):
                torch.nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    torch.nn.init.constant_(m.bias, 0)
            elif isinstance(m, torch.nn.Linear):
                m.weight.data.normal_(0.0, std=0.01)
                if m.bias is not None:
                    torch.nn.init.constant_(m.bias, 0)
            elif isinstance(m, torch.nn.BatchNorm2d):
                torch.nn.init.constant_(m.weight, 1)
                torch.nn.init.constant_(m.bias, 0)
"""
    model_culane_text = model_culane_text.replace('from utils.common import initialize_weights', init_helper)
model_culane_text = model_culane_text.replace(
    ').cuda()',
    ').to(torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\"))',
)
model_culane_py.write_text(model_culane_text, encoding='utf-8')

sys.path.insert(0, str(UFLD_ROOT))
from utils.config import Config
from model.model_culane import get_model
import torchvision.transforms as T



def pred2coords(pred, row_anchor, col_anchor, local_width=1, original_image_width=1640, original_image_height=590):
    batch_size, num_grid_row, num_cls_row, num_lane_row = pred['loc_row'].shape
    batch_size, num_grid_col, num_cls_col, num_lane_col = pred['loc_col'].shape
    max_indices_row = pred['loc_row'].argmax(1).cpu()
    valid_row = pred['exist_row'].argmax(1).cpu()
    max_indices_col = pred['loc_col'].argmax(1).cpu()
    valid_col = pred['exist_col'].argmax(1).cpu()
    pred['loc_row'] = pred['loc_row'].cpu()
    pred['loc_col'] = pred['loc_col'].cpu()
    coords = []
    row_lane_idx = [1, 2]
    col_lane_idx = [0, 3]

    for i in row_lane_idx:
        lane = []
        if valid_row[0, :, i].sum() > num_cls_row / 2:
            for k in range(valid_row.shape[1]):
                if valid_row[0, k, i]:
                    all_ind = torch.tensor(list(range(
                        max(0, max_indices_row[0, k, i] - local_width),
                        min(num_grid_row - 1, max_indices_row[0, k, i] + local_width) + 1,
                    )))
                    out_tmp = (pred['loc_row'][0, all_ind, k, i].softmax(0) * all_ind.float()).sum() + 0.5
                    out_tmp = out_tmp / (num_grid_row - 1) * original_image_width
                    lane.append((int(out_tmp), int(row_anchor[k] * original_image_height)))
        coords.append(lane)

    for i in col_lane_idx:
        lane = []
        if valid_col[0, :, i].sum() > num_cls_col / 4:
            for k in range(valid_col.shape[1]):
                if valid_col[0, k, i]:
                    all_ind = torch.tensor(list(range(
                        max(0, max_indices_col[0, k, i] - local_width),
                        min(num_grid_col - 1, max_indices_col[0, k, i] + local_width) + 1,
                    )))
                    out_tmp = (pred['loc_col'][0, all_ind, k, i].softmax(0) * all_ind.float()).sum() + 0.5
                    out_tmp = out_tmp / (num_grid_col - 1) * original_image_height
                    lane.append((int(col_anchor[k] * original_image_width), int(out_tmp)))
        coords.append(lane)
    return coords

ufld_cfg = Config.fromfile(str(UFLD_ROOT / 'configs' / 'culane_res18.py'))
ufld_cfg.batch_size = 1
ufld_cfg.test_model = str(ufld_ckpt)
ufld_cfg.row_anchor = np.linspace(0.42, 1, ufld_cfg.num_row)
ufld_cfg.col_anchor = np.linspace(0, 1, ufld_cfg.num_col)

lane_teacher = get_model(ufld_cfg)
state = torch.load(str(ufld_ckpt), map_location='cpu')
state_dict = state.get('model', state)
state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
lane_teacher.load_state_dict(state_dict, strict=False)
lane_teacher = lane_teacher.to(DEVICE).eval()

lane_transform = T.Compose([
    T.Resize((int(ufld_cfg.train_height / ufld_cfg.crop_ratio), ufld_cfg.train_width)),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
print('Teachers loaded.')

## 4 - Label Helpers

In [ ]:
YOLOPV2_LABELS = [
    'car', 'truck', 'bus', 'person', 'rider',
    'bicycle', 'motorcycle', 'traffic light', 'traffic sign', 'train',
]
YOLOPV2_CLASS_TO_ID = {name: idx for idx, name in enumerate(YOLOPV2_LABELS)}

# COCO -> YOLOPv2/BDD-style driving classes. COCO has no direct rider class.
COCO_TO_YOLOPV2 = {
    'person': 'person',
    'bicycle': 'bicycle',
    'car': 'car',
    'motorcycle': 'motorcycle',
    'bus': 'bus',
    'train': 'train',
    'truck': 'truck',
    'traffic light': 'traffic light',
    'stop sign': 'traffic sign',
}


def resize_square(frame_bgr, size=IMAGE_SIZE):
    return cv2.resize(frame_bgr, (size, size), interpolation=cv2.INTER_AREA)


def yolo_box_teacher(frame_bgr, conf=YOLO_CONF):
    result = box_teacher.predict(frame_bgr, imgsz=640, conf=conf, verbose=False, device=0 if DEVICE == 'cuda' else 'cpu')[0]
    detections = []
    names = result.names
    if result.boxes is None:
        return detections
    for box in result.boxes:
        coco_name = names[int(box.cls.item())]
        mapped = COCO_TO_YOLOPV2.get(coco_name)
        if mapped is None:
            continue
        x1, y1, x2, y2 = box.xyxy[0].detach().cpu().numpy().astype(float).tolist()
        detections.append({
            'cls': YOLOPV2_CLASS_TO_ID[mapped],
            'label': mapped,
            'conf': float(box.conf.item()),
            'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
        })
    return detections


@torch.inference_mode()
def segformer_drivable_mask(frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    inputs = seg_processor(images=rgb, return_tensors='pt').to(DEVICE)
    logits = seg_model(**inputs).logits
    logits = F.interpolate(logits, size=rgb.shape[:2], mode='bilinear', align_corners=False)
    pred = logits.argmax(dim=1)[0].detach().cpu().numpy().astype(np.uint8)
    mask = np.isin(pred, ROAD_CLASS_IDS).astype(np.uint8) * 255
    return mask


@torch.inference_mode()
def ufld_lane_mask(frame_bgr, thickness=5):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    pil = Image.fromarray(rgb)
    tensor = lane_transform(pil).unsqueeze(0).to(DEVICE)
    pred = lane_teacher(tensor)
    coords = pred2coords(
        pred,
        ufld_cfg.row_anchor,
        ufld_cfg.col_anchor,
        original_image_width=w,
        original_image_height=h,
    )
    mask = np.zeros((h, w), dtype=np.uint8)
    for lane in coords:
        pts = [(int(x), int(y)) for x, y in lane if 0 <= int(x) < w and 0 <= int(y) < h]
        if len(pts) >= 2:
            cv2.polylines(mask, [np.array(pts, dtype=np.int32)], isClosed=False, color=255, thickness=thickness)
        for pt in pts:
            cv2.circle(mask, pt, max(2, thickness // 2), 255, -1)
    return mask


def write_yolo_labels(detections, out_path: Path, orig_w: int, orig_h: int):
    lines = []
    for det in detections:
        x1 = np.clip(det['x1'], 0, orig_w - 1)
        y1 = np.clip(det['y1'], 0, orig_h - 1)
        x2 = np.clip(det['x2'], 0, orig_w - 1)
        y2 = np.clip(det['y2'], 0, orig_h - 1)
        bw = max(float(x2 - x1), 1.0)
        bh = max(float(y2 - y1), 1.0)
        cx = (float(x1) + bw * 0.5) / orig_w
        cy = (float(y1) + bh * 0.5) / orig_h
        lines.append(f"{int(det['cls'])} {cx:.6f} {cy:.6f} {bw / orig_w:.6f} {bh / orig_h:.6f}")
    out_path.write_text('\n'.join(lines), encoding='utf-8')


def save_labeled_frame(frame_bgr, stem, source_domain, source_name, frame_index=None):
    h, w = frame_bgr.shape[:2]
    detections = yolo_box_teacher(frame_bgr)
    lane_mask = ufld_lane_mask(frame_bgr)
    drivable_mask = segformer_drivable_mask(frame_bgr)

    img_320 = resize_square(frame_bgr)
    lane_320 = cv2.resize(lane_mask, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST)
    driv_320 = cv2.resize(drivable_mask, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST)

    img_path = OUT_IMAGES / f'{stem}.jpg'
    label_path = OUT_LABELS / f'{stem}.txt'
    lane_path = OUT_LANE / f'{stem}.png'
    drivable_path = OUT_DRIVABLE / f'{stem}.png'

    cv2.imwrite(str(img_path), img_320)
    cv2.imwrite(str(lane_path), lane_320)
    cv2.imwrite(str(drivable_path), driv_320)
    write_yolo_labels(detections, label_path, w, h)

    return {
        'image_path': str(img_path),
        'label_path': str(label_path),
        'lane_mask_path': str(lane_path),
        'drivable_mask_path': str(drivable_path),
        'domain': source_domain,
        'is_real': int(source_domain == 'real'),
        'source_name': source_name,
        'frame_index': frame_index if frame_index is not None else '',
        'width': IMAGE_SIZE,
        'height': IMAGE_SIZE,
        'num_objects': len(detections),
    }

## 5 - Real Video Sampling Plan (~5000 frames)

In [ ]:
def video_meta(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open video: {video_path}')
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    cap.release()
    return {'path': video_path, 'fps': fps, 'frames': frames, 'width': w, 'height': h, 'duration_s': frames / fps if fps else 0}

video_infos = [video_meta(p) for p in real_videos]
total_video_frames = sum(v['frames'] for v in video_infos)
remaining = TARGET_REAL_FRAMES
sampling_plan = []
for i, info in enumerate(video_infos):
    if i == len(video_infos) - 1:
        target = remaining
    else:
        target = int(round(TARGET_REAL_FRAMES * info['frames'] / max(total_video_frames, 1)))
        remaining -= target
    target = min(target, info['frames'])
    sampled_fps = target / max(info['duration_s'], 1e-6)
    sampling_plan.append({**info, 'target_frames': target, 'sampled_fps': sampled_fps})

plan_df = pd.DataFrame([{k: str(v) if isinstance(v, Path) else v for k, v in row.items()} for row in sampling_plan])
display(plan_df[['path', 'frames', 'fps', 'duration_s', 'width', 'height', 'target_frames', 'sampled_fps']])
print('Total sampled real frames:', sum(p['target_frames'] for p in sampling_plan))

## 6 - Build Final Dataset

In [ ]:
def iter_selected_video_frames(video_path: Path, target_count: int):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if total <= 0 or target_count <= 0:
        cap.release()
        return
    indices = np.unique(np.linspace(0, total - 1, num=target_count, dtype=np.int64))
    for frame_idx in tqdm(indices, desc=video_path.name):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame = cap.read()
        if ok:
            yield int(frame_idx), frame
    cap.release()

rows = []

# CARLA synthetic frames.
for img_path in tqdm(carla_images, desc='CARLA images'):
    frame = cv2.imread(str(img_path))
    if frame is None:
        continue
    stem = f'carla_{img_path.stem}'
    rows.append(save_labeled_frame(frame, stem, 'simulated', img_path.name))

# Real dashcam frames sampled to approximately TARGET_REAL_FRAMES total.
for info in sampling_plan:
    video_path = info['path']
    for frame_idx, frame in iter_selected_video_frames(video_path, info['target_frames']):
        stem = f'real_{video_path.stem}_{frame_idx:06d}'
        rows.append(save_labeled_frame(frame, stem, 'real', video_path.name, frame_idx))

manifest = pd.DataFrame(rows)
manifest_path = OUT_ROOT / 'manifest.csv'
manifest.to_csv(manifest_path, index=False)
print(f'Saved {len(manifest)} labeled frames to {OUT_ROOT}')
print(f'Manifest: {manifest_path}')
display(manifest.groupby('domain').agg(frames=('image_path', 'count'), objects=('num_objects', 'sum')).reset_index())

## 7 - Visual QA Samples

In [ ]:
def read_yolo_label_file(label_path: Path, img_w=IMAGE_SIZE, img_h=IMAGE_SIZE):
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        cls, cx, cy, bw, bh = map(float, parts)
        x1 = int((cx - bw / 2) * img_w)
        y1 = int((cy - bh / 2) * img_h)
        x2 = int((cx + bw / 2) * img_w)
        y2 = int((cy + bh / 2) * img_h)
        boxes.append((int(cls), x1, y1, x2, y2))
    return boxes


def overlay_sample(row):
    img = cv2.imread(row.image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lane = cv2.imread(row.lane_mask_path, cv2.IMREAD_GRAYSCALE)
    drivable = cv2.imread(row.drivable_mask_path, cv2.IMREAD_GRAYSCALE)
    vis = img.copy()
    vis[drivable > 0] = (0.55 * vis[drivable > 0] + np.array([0, 180, 70]) * 0.45).astype(np.uint8)
    vis[lane > 0] = np.array([0, 220, 255], dtype=np.uint8)
    for cls, x1, y1, x2, y2 in read_yolo_label_file(Path(row.label_path)):
        cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 220, 0), 2)
        cv2.putText(vis, YOLOPV2_LABELS[cls], (x1, max(12, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 220, 0), 1)
    return vis

sample_rows = pd.concat([
    manifest[manifest.domain == 'simulated'].sample(min(4, (manifest.domain == 'simulated').sum()), random_state=7),
    manifest[manifest.domain == 'real'].sample(min(4, (manifest.domain == 'real').sum()), random_state=7),
])

fig, axes = plt.subplots(len(sample_rows), 1, figsize=(6, 3 * len(sample_rows)))
if len(sample_rows) == 1:
    axes = [axes]
for ax, (_, row) in zip(axes, sample_rows.iterrows()):
    vis = overlay_sample(row)
    ax.imshow(vis)
    ax.set_title(f"{row.domain}: {Path(row.image_path).name} | objects={row.num_objects}")
    ax.axis('off')
    cv2.imwrite(str(OUT_QA / f"qa_{Path(row.image_path).stem}.jpg"), cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
plt.tight_layout()
plt.show()
print('QA overlays saved to', OUT_QA)

## 8 - Final Layout

The generated dataset is ready for `notebooks/03_finetune_yolopv2.ipynb`:

```text
data/yolopv2_finetune/
  images/*.jpg              # 320x320 RGB frames from CARLA and real videos
  labels/*.txt              # YOLO format boxes in YOLOPv2 class order
  masks/lane/*.png          # UFLD-v2 lane masks, 0/255
  masks/drivable/*.png      # SegFormer road/drivable masks, 0/255
  manifest.csv              # paths + domain/is_real metadata
  qa_samples/*.jpg          # visual inspection overlays
```

Important: this notebook intentionally does not use YOLOPv2 for pseudo-labeling; YOLOPv2 only consumes these labels later during finetuning.